# Model evaluation — proxy metrics

Proxy evaluation for the AI Creative Auditor (no ground-truth LLM labels).

Run from repo root first: `python init_db.py` and `python run_pipeline.py`

In [ ]:
import json
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("..").resolve()
DB_PATH = ROOT / "ad_database.duckdb"

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found at {DB_PATH}. Run: python init_db.py && python run_pipeline.py"
    )

conn = duckdb.connect(str(DB_PATH), read_only=True)
df = conn.execute("SELECT * FROM ad_evaluations ORDER BY created_at").df()
conn.close()

df["total_score"] = df["design_score"] + df["business_score"]
print(f"Loaded {len(df)} evaluations")
df.head()

## 1. OCR correction rate (proxy)

In [ ]:
def was_corrected(row):
    raw = str(row.get("raw_ocr_text", "")).strip()
    clean = str(row.get("corrected_text", "")).strip()
    if clean in ("None", "", "nan"):
        return False
    return raw != clean and raw not in ("No text found", "")

df["ocr_corrected"] = df.apply(was_corrected, axis=1)
correction_rate = df["ocr_corrected"].mean() if len(df) else 0
print(f"OCR correction rate: {correction_rate:.1%} ({df['ocr_corrected'].sum()}/{len(df)})")

## 2. Score distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col in zip(axes, ["design_score", "business_score", "total_score"]):
    ax.hist(df[col].dropna(), bins=range(1, 22), edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel("score")
plt.tight_layout()
plt.show()

print(df[["design_score", "business_score", "total_score"]].describe())

## 3. People count vs scores

In [ ]:
if "person_count" in df.columns:
    seg = df.groupby("person_count")[["design_score", "business_score", "total_score"]].mean()
    display(seg)
    seg["total_score"].plot(kind="bar", title="Avg total score by people count")
    plt.ylabel("avg total score")
    plt.show()

## 4. Color psychology mix

In [ ]:
tags = []
for val in df.get("color_hex_json", pd.Series()).dropna():
    try:
        colors = json.loads(val) if isinstance(val, str) else val
        if colors:
            tags.append(colors[0].get("psychology", "unknown"))
    except (json.JSONDecodeError, TypeError):
        pass

if tags:
    psych = pd.Series(tags).value_counts()
    psych.plot(kind="pie", autopct="%1.0f%%", title="Dominant color psychology")
    plt.ylabel("")
    plt.show()
else:
    print("No color_hex_json data — re-run pipeline with schema v2")

## 5. Limitations (for interviews)

- LLM scores are **subjective proxies**, not validated against human creative directors.
- Temperature 0.8 introduces run-to-run variance.
- WCAG column is heuristic on OCR boxes only.
- Recommend human spot-check on 10–20 ads before production use.